# Model Merging with xaytune

**Model merging** combines the weights of multiple fine-tuned models into a single model without additional training. This is useful when:

- You have multiple task-specific fine-tunes and want a multi-task model
- You want to combine capabilities from different training runs
- You need to balance performance across different domains

xaytune supports four merging methods:

1. **Linear**: Weighted average of model weights — simple, fast, effective for similar models
2. **SLERP**: Spherical interpolation — preserves weight magnitude, smoother interpolation
3. **TIES**: Trim/elect-sign/merge — resolves weight conflicts, good for many models
4. **DARE**: Drop-and-rescale — stochastic sparsification, reduces interference

This notebook demonstrates each method using synthetic tensors.

## Setup: Create Fake State Dicts

We'll simulate a base model and two fine-tuned variants by adding small perturbations.

In [ ]:
import torch

torch.manual_seed(42)
base_sd = {
    "attn.weight": torch.randn(8, 8),
    "mlp.weight": torch.randn(8, 8),
    "ln.bias": torch.randn(8)
}
model_a = {k: v + torch.randn_like(v) * 0.1 for k, v in base_sd.items()}
model_b = {k: v + torch.randn_like(v) * 0.1 for k, v in base_sd.items()}
print(f"Created base model and 2 fine-tuned variants")
print(f"Keys: {list(base_sd.keys())}")

## Linear Merge

**Linear merge** computes a weighted average of model weights:

```
merged = w1 * model_a + w2 * model_b
```

This is the simplest method and works well when models are similar. The weights should sum to 1.0.

In [ ]:
from xaytune.export.model_merge import _linear_merge

merged = _linear_merge([model_a, model_b], weights=[0.6, 0.4])
print("Linear merge with weights [0.6, 0.4]:\n")
for key in merged:
    diff_a = (merged[key] - model_a[key]).abs().mean().item()
    diff_b = (merged[key] - model_b[key]).abs().mean().item()
    print(f"{key}: avg diff from A={diff_a:.4f}, from B={diff_b:.4f}")

## SLERP Merge

**SLERP (Spherical Linear Interpolation)** interpolates along the surface of a hypersphere:

```
merged = slerp(model_a, model_b, t)
```

Unlike linear interpolation, SLERP preserves weight magnitude and provides smoother transitions. The parameter `t` ranges from 0.0 (all model_a) to 1.0 (all model_b).

In [ ]:
from xaytune.export.model_merge import _slerp_merge

print("SLERP merge with varying t values:\n")
for t_val in [0.0, 0.25, 0.5, 0.75, 1.0]:
    merged = _slerp_merge(model_a, model_b, t=t_val)
    diff_a = sum((merged[k] - model_a[k]).abs().mean().item() for k in merged) / len(merged)
    diff_b = sum((merged[k] - model_b[k]).abs().mean().item() for k in merged) / len(merged)
    print(f"t={t_val:.2f}: dist_to_A={diff_a:.4f}, dist_to_B={diff_b:.4f}")

## TIES Merge

**TIES (Trim, Elect Sign, Merge)** resolves weight conflicts when merging multiple models:

1. **Trim**: Keep only the top-k% largest weight changes (task vectors)
2. **Elect Sign**: Resolve sign conflicts by majority vote
3. **Merge**: Average the aligned task vectors

The `density` parameter controls what fraction of weights to keep (0.0-1.0). Lower density = more aggressive pruning.

In [ ]:
from xaytune.export.model_merge import _ties_merge

print("TIES merge with varying density:\n")
for density in [0.3, 0.5, 0.7, 1.0]:
    merged = _ties_merge([model_a, model_b], base_sd, density=density, weight=1.0)
    diff = sum((merged[k] - base_sd[k]).abs().mean().item() for k in merged) / len(merged)
    print(f"density={density}: avg task vector magnitude={diff:.4f}")

## DARE Merge

**DARE (Drop And REscale)** applies stochastic pruning to task vectors:

1. Randomly drop weight changes with probability (1 - density)
2. Rescale remaining weights to maintain expected magnitude
3. Merge the sparsified task vectors

This reduces interference between models while maintaining overall performance. The `density` parameter controls the drop rate.

In [ ]:
from xaytune.export.model_merge import _dare_merge

print("DARE merge with varying density:\n")
for density in [0.3, 0.5, 0.7, 1.0]:
    merged = _dare_merge([model_a, model_b], base_sd, density=density, weight=1.0, seed=42)
    diff = sum((merged[k] - base_sd[k]).abs().mean().item() for k in merged) / len(merged)
    print(f"density={density}: avg task vector magnitude={diff:.4f}")

## Real-World Usage

The high-level `model_merge()` API handles loading models, merging, and saving automatically.

In [ ]:
# Real-world usage with actual model checkpoints:
#
# from xaytune.export.model_merge import model_merge
#
# result = model_merge(
#     models=["output/ft-coding", "output/ft-math"],
#     method="ties",
#     base_model="meta-llama/Llama-3.1-8B",
#     density=0.5,
#     output="output/merged",
# )
# print(result.summary())
print("See commented code above for real-world model_merge() usage.")
print("Requires actual model checkpoints on disk.")

## CLI Reference

xaytune provides a CLI for merging models:

```bash
# Linear merge
xaytune merge \
  --models output/ft-coding output/ft-math \
  --method linear \
  --weights 0.6 0.4 \
  --output output/merged-linear

# SLERP merge
xaytune merge \
  --models output/ft-coding output/ft-math \
  --method slerp \
  --t 0.5 \
  --output output/merged-slerp

# TIES merge
xaytune merge \
  --models output/ft-coding output/ft-math output/ft-reasoning \
  --method ties \
  --base-model meta-llama/Llama-3.1-8B \
  --density 0.5 \
  --weight 1.0 \
  --output output/merged-ties

# DARE merge
xaytune merge \
  --models output/ft-coding output/ft-math output/ft-reasoning \
  --method dare \
  --base-model meta-llama/Llama-3.1-8B \
  --density 0.7 \
  --weight 1.0 \
  --seed 42 \
  --output output/merged-dare
```

## Next Steps

- **Train models to merge**: See [02_finetuning.ipynb](02_finetuning.ipynb) for training task-specific models
- **Experiment with methods**: Try different merge methods and parameters on your models
- **Evaluate merged models**: Test the merged model on your target tasks
- **Iterate**: Adjust density/weight parameters based on evaluation results

For more details, see the [xaytune documentation](https://github.com/szaher/xaytune).